# 04. 전이학습 — ResNet18 (frozen vs fine-tuning)

`03`의 scratch CNN은 val 정확도 64%에 그쳤습니다. 여기서는 ImageNet으로 사전학습된
ResNet18을 가져와 두 가지 방식으로 학습시키고, scratch와 비교합니다.

| 실험 | backbone | 학습 대상 | 의미 |
|---|---|---|---|
| `frozen` | 동결 | 마지막 FC만 | ImageNet 특징을 **그대로** 재사용 |
| `finetune` | 학습 | 전체 (backbone은 낮은 lr) | 쓰레기 이미지에 맞게 **재조정** |

**강의와의 관계**

`4_2_ResNet`, `4_3_EfficientNet`에서는 `include_top=True`로 ImageNet 1000클래스
분류기를 그대로 붙여 **추론만** 했습니다(개 사진 → `miniature_poodle`).
여기서는 그 마지막 층을 떼어내고 12클래스 분류기로 갈아끼운 뒤 재학습합니다.
강의가 멈춘 지점의 바로 다음 단계입니다.

**불균형 대응은 `sampler`로 고정합니다.** `03`에서 세 방식을 비교해
최악 클래스 recall 0.181 → 0.304, 편차 0.217 → 0.181로 가장 나았기 때문입니다.
여기서 또 세 번 돌릴 이유가 없습니다.

## 1. 환경

In [1]:
import json
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch import amp
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
import torchvision.models as models
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import (confusion_matrix, classification_report,
                             f1_score, recall_score)

assert torch.cuda.is_available(), "CUDA 미탐지. 커널이 .venv인지 확인하세요."
device = torch.device("cuda")
torch.backends.cudnn.benchmark = True

sns.set_theme(style="whitegrid")
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

print("torch :", torch.__version__)
print("GPU   :", torch.cuda.get_device_name(0))
print("VRAM  : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1024**3))

torch : 2.11.0+cu128
GPU   : NVIDIA GeForce RTX 5060 Laptop GPU
VRAM  : 8.0 GB


## 2. 설정

**`IMG_SIZE = 224`** — `03`의 128과 다릅니다. ResNet18의 사전학습 가중치가
224 입력으로 학습되었기 때문입니다. 보고서에서 scratch와 비교할 때
"입력 크기도 달랐다"는 점을 명시하세요. 순수한 아키텍처 비교가 아닙니다.

**정규화는 ImageNet 통계를 씁니다.** 사전학습 가중치가 그 분포를 전제로 학습되었으므로,
데이터셋 자체 통계를 쓰면 오히려 손해입니다. (`03`은 scratch였으므로 데이터셋 통계 사용)

In [ ]:
OUT = Path("outputs/garbage")
(OUT / "models").mkdir(parents=True, exist_ok=True)

IMG_SIZE     = 224
BATCH_SIZE   = 64      # OOM 나면 32로
EPOCHS       = 12
LR_HEAD      = 1e-3    # 새로 만든 분류기 — 랜덤 초기화라 크게
LR_BACKBONE  = 1e-4    # 사전학습층 — 이미 좋은 가중치라 조심스럽게
NUM_WORKERS  = 0       # 03에서 잘 돌아간 값으로 맞추세요
SEED         = 42

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

split     = pd.read_csv(OUT / "metrics" / "split.csv")
label_map = json.load(open(OUT / "metrics" / "label_map.json", encoding="utf-8"))
stats     = json.load(open(OUT / "metrics" / "norm_stats.json", encoding="utf-8"))
cw_df     = pd.read_csv(OUT / "metrics" / "class_weights.csv").sort_values("label_idx")

classes   = [c for c, _ in sorted(label_map.items(), key=lambda kv: kv[1])]
N_CLASSES = len(classes)

train_df = split[split.split == "train"].reset_index(drop=True)
val_df   = split[split.split == "val"].reset_index(drop=True)

mean, std = stats["imagenet_mean"], stats["imagenet_std"]   # ImageNet 통계
print(f"train {len(train_df):,} | val {len(val_df):,} | classes {N_CLASSES}")
print("정규화(ImageNet):", mean, std)

## 3. 데이터

증강 방침은 `03`과 동일합니다. EDA에서 유리 3종의 Hue 겹침 계수가 0.5 미만
(brown-green 0.146, green-white 0.202, brown-white 0.422)이었으므로
**색이 판별 신호**입니다. `saturation`은 0.05, `hue`는 0으로 고정합니다.

`03`의 과소적합 진단을 반영해 crop scale과 회전도 완화된 값으로 시작합니다.

In [3]:
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.05, hue=0.0),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])


class GarbageDataset(Dataset):
    def __init__(self, df, transform):
        self.paths     = df["path"].tolist()
        self.targets   = df["label_idx"].tolist()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")   # P모드 34장 대응
        return self.transform(img), self.targets[i]


train_ds = GarbageDataset(train_df, train_tf)
val_ds   = GarbageDataset(val_df,   eval_tf)

# 03의 결론에 따라 sampler 고정
per_class = cw_df.set_index("label_idx")["class_weight"].to_dict()
sample_w  = [per_class[t] for t in train_ds.targets]

val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

x, y = train_ds[0]
print("샘플 텐서:", x.shape, "| 라벨:", y, classes[y])

NameError: name 'IMG_SIZE' is not defined

## 4. 모델 — 마지막 층 교체

```python
m.fc = nn.Linear(512, 12)
```

이 한 줄이 강의의 `include_top=True`와 갈리는 지점입니다.
ImageNet 1000클래스 분류기를 떼어내고 12클래스용 새 층을 답니다.
새 층은 랜덤 초기화 상태이므로 반드시 학습되어야 합니다.

### frozen 모드에서 BatchNorm 처리

`requires_grad=False`만으로는 동결이 완전하지 않습니다. BatchNorm은 학습 파라미터와
별개로 **running_mean/var라는 통계를 train 모드에서 계속 갱신**하기 때문입니다.
그대로 두면 backbone이 조금씩 변해서 "ImageNet 특징을 그대로 쓴다"는 전제가 깨집니다.

그래서 `frozen`일 때는 backbone의 BN 모듈만 `eval()`로 유지합니다.

In [ ]:
def build_model(mode):
    """mode: 'frozen' | 'finetune'"""
    m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, N_CLASSES)     # 1000 → 12

    if mode == "frozen":
        for name, p in m.named_parameters():
            p.requires_grad = name.startswith("fc.")

    return m.to(device)


def set_train_mode(model, mode):
    """frozen이면 backbone BN의 통계 갱신을 막는다"""
    model.train()
    if mode == "frozen":
        for mod in model.modules():
            if isinstance(mod, nn.BatchNorm2d):
                mod.eval()


for mode in ["frozen", "finetune"]:
    m = build_model(mode)
    total = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"{mode:9s} 전체 {total:,} | 학습 대상 {trainable:,} "
          f"({100 * trainable / total:.1f}%)")
    del m
torch.cuda.empty_cache()

## 5. 학습·평가 루프

`03`과 동일한 구조입니다. AMP 적용, `break` 없음, 선택 기준은 macro-F1.

In [ ]:
def train_epoch(model, loader, criterion, optimizer, scaler, mode):
    set_train_mode(model, mode)
    loss_sum, correct, total = 0.0, 0, 0

    for inputs, labels in loader:
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with amp.autocast("cuda", dtype=torch.float16):
            outputs = model(inputs)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        loss_sum += loss.item() * labels.size(0)
        correct  += outputs.argmax(1).eq(labels).sum().item()
        total    += labels.size(0)

    return loss_sum / total, 100 * correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    loss_sum, total = 0.0, 0
    preds, trues = [], []

    for inputs, labels in loader:
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with amp.autocast("cuda", dtype=torch.float16):
            outputs = model(inputs)
            loss = criterion(outputs, labels)

        loss_sum += loss.item() * labels.size(0)
        total    += labels.size(0)
        preds.append(outputs.argmax(1).cpu())
        trues.append(labels.cpu())

    y_pred = torch.cat(preds).numpy()
    y_true = torch.cat(trues).numpy()

    return {"loss": loss_sum / total,
            "acc": 100 * (y_pred == y_true).mean(),
            "macro_f1": f1_score(y_true, y_pred, average="macro"),
            "y_pred": y_pred, "y_true": y_true}

## 6. 실험 러너

**옵티마이저에서 두 방식이 갈립니다.**

- `frozen`: FC 파라미터만 넘김. 나머지는 `requires_grad=False`라 넘겨봐야 의미 없음
- `finetune`: 파라미터 그룹을 둘로 나눠 **서로 다른 학습률** 적용
  - backbone `1e-4` — 이미 좋은 가중치라 크게 흔들면 사전학습 효과가 날아갑니다
  - head `1e-3` — 랜덤 초기화라 빠르게 학습되어야 합니다

이걸 discriminative learning rate라고 하며, 전이학습의 기본 기법입니다.

In [ ]:
def run_transfer(mode, epochs=EPOCHS):
    set_seed(SEED)
    model = build_model(mode)

    sampler = WeightedRandomSampler(sample_w, num_samples=len(sample_w),
                                    replacement=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    criterion = nn.CrossEntropyLoss()

    if mode == "frozen":
        optimizer = optim.AdamW(model.fc.parameters(), lr=LR_HEAD, weight_decay=1e-4)
    else:
        backbone = [p for n, p in model.named_parameters() if not n.startswith("fc.")]
        optimizer = optim.AdamW(
            [{"params": backbone,               "lr": LR_BACKBONE},
             {"params": model.fc.parameters(),  "lr": LR_HEAD}],
            weight_decay=1e-4)

    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
    scaler    = amp.GradScaler("cuda")

    ckpt = OUT / "models" / f"transfer_{mode}.pt"
    best_f1, history = -1.0, []
    t0 = time.time()

    print(f"\n{'=' * 70}\n[resnet18 / {mode}]  epochs={epochs}  "
          f"batches/epoch={len(train_loader)}\n{'=' * 70}")

    for ep in range(1, epochs + 1):
        tr_loss, tr_acc = train_epoch(model, train_loader, criterion,
                                      optimizer, scaler, mode)
        va = evaluate(model, val_loader, criterion)
        scheduler.step()

        history.append({"epoch": ep, "train_loss": tr_loss, "train_acc": tr_acc,
                        "val_loss": va["loss"], "val_acc": va["acc"],
                        "val_macro_f1": va["macro_f1"]})

        star = ""
        if va["macro_f1"] > best_f1:
            best_f1 = va["macro_f1"]
            torch.save({"model": model.state_dict(), "mode": mode, "epoch": ep,
                        "val_macro_f1": best_f1, "classes": classes,
                        "img_size": IMG_SIZE}, ckpt)
            star = "  *best"

        print(f"  ep{ep:02d}  train {tr_loss:.3f}/{tr_acc:5.1f}%  "
              f"val {va['loss']:.3f}/{va['acc']:5.1f}%  macroF1 {va['macro_f1']:.4f}"
              f"  gap {tr_acc - va['acc']:+5.1f}p{star}")

    elapsed = time.time() - t0
    model.load_state_dict(torch.load(ckpt)["model"])
    final = evaluate(model, val_loader, criterion)
    recalls = recall_score(final["y_true"], final["y_pred"],
                           average=None, labels=range(N_CLASSES), zero_division=0)

    hist = pd.DataFrame(history)
    hist.to_csv(OUT / "metrics" / f"history_transfer_{mode}.csv",
                index=False, encoding="utf-8")

    print(f"  완료 {elapsed/60:.1f}분 | best macroF1 {best_f1:.4f} "
          f"(ep{int(hist['val_macro_f1'].idxmax()) + 1})")

    del model
    torch.cuda.empty_cache()

    return {"strategy": f"resnet18_{mode}", "history": hist, "final": final,
            "recalls": recalls, "minutes": elapsed / 60, "best_f1": best_f1}

## 7. 실행

먼저 1에폭으로 시간과 VRAM을 확인하세요.
`torch.cuda.OutOfMemoryError`가 나면 `BATCH_SIZE`를 32로 낮추고 커널을 재시작하세요.

In [2]:
_probe = run_transfer("frozen", epochs=1)
print(f"\nfrozen 1에폭 = {_probe['minutes']:.1f}분")
print("VRAM peak : %.2f GB" % (torch.cuda.max_memory_allocated() / 1024**3))
del _probe
torch.cuda.reset_peak_memory_stats()

NameError: name 'run_transfer' is not defined

In [ ]:
results = {}
for mode in ["frozen", "finetune"]:
    results[mode] = run_transfer(mode)

## 8. 비교 — scratch 포함

In [ ]:
rows = []
for m, r in results.items():
    f = r["final"]
    rows.append({"model": f"resnet18_{m}",
                 "val_acc":    round(f["acc"], 2),
                 "macro_f1":   round(f["macro_f1"], 4),
                 "min_recall": round(float(r["recalls"].min()), 3),
                 "worst_class": classes[int(r["recalls"].argmin())],
                 "recall_std": round(float(r["recalls"].std()), 3),
                 "minutes":    round(r["minutes"], 1)})

transfer_summary = pd.DataFrame(rows)

# 03의 sampler 결과를 같은 표에 붙인다
base = pd.read_csv(OUT / "metrics" / "baseline_comparison.csv")
base_row = base[base["strategy"] == "sampler"].iloc[0]
scratch = pd.DataFrame([{
    "model": "GarbageCNN_scratch", "val_acc": base_row["val_acc"],
    "macro_f1": base_row["macro_f1"], "min_recall": base_row["min_recall"],
    "worst_class": base_row["worst_class"], "recall_std": base_row["recall_std"],
    "minutes": base_row["minutes"]}])

compare = pd.concat([scratch, transfer_summary], ignore_index=True)
compare.to_csv(OUT / "metrics" / "transfer_comparison.csv",
               index=False, encoding="utf-8")
print(compare.to_string(index=False))

**읽는 법**

- `frozen`이 scratch를 크게 넘긴다면, **ImageNet에서 배운 특징이 쓰레기 이미지에도
  그대로 통한다**는 증거입니다. backbone은 한 번도 이 데이터를 본 적이 없습니다.
- `finetune`이 `frozen`보다 높다면, 재조정이 필요한 도메인 차이가 있었다는 뜻입니다.
  차이가 작다면 ImageNet 특징만으로 충분했다는 뜻이고, 그것도 유효한 결론입니다.
- `minutes`도 비교하세요. `frozen`은 gradient가 마지막 층에만 흐르므로 더 빠릅니다.
  성능이 비슷한데 시간이 절반이면 실무적으로는 `frozen`이 낫습니다.

In [ ]:
rec = pd.DataFrame({m: results[m]["recalls"] for m in results}, index=classes)
rec.insert(0, "val_n",
           val_df["label_idx"].value_counts().reindex(range(N_CLASSES)).values)
print("클래스별 recall (val)")
print(rec.round(3).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x, width = np.arange(N_CLASSES), 0.38
colors = {"frozen": "#4a7ba7", "finetune": "#c0392b"}

for i, m in enumerate(results):
    ax.bar(x + (i - 0.5) * width, results[m]["recalls"], width,
           label=f"resnet18_{m}", color=colors[m])

ax.set_xticks(x); ax.set_xticklabels(classes, rotation=45, ha="right")
ax.set_ylabel("recall (val)"); ax.set_ylim(0, 1)
ax.set_title("Per-class recall — frozen vs fine-tuning")
ax.axhline(0.5, ls="--", lw=1, color="gray")
ax.legend()
plt.tight_layout()
plt.savefig(OUT / "figures" / "11_transfer_recall.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, m in zip(axes, results):
    h = results[m]["history"]
    ax.plot(h["epoch"], h["train_acc"], label="train acc", color="#4a7ba7")
    ax.plot(h["epoch"], h["val_acc"],   label="val acc",   color="#c0392b")
    ax.plot(h["epoch"], h["val_macro_f1"] * 100, label="val macroF1x100",
            color="#27ae60", ls="--")
    ax.set_title(f"resnet18_{m}"); ax.set_xlabel("epoch"); ax.set_ylim(0, 100)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUT / "figures" / "12_transfer_curves.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, m in zip(axes, results):
    f = results[m]["final"]
    cm = confusion_matrix(f["y_true"], f["y_pred"], labels=range(N_CLASSES))
    cmn = cm / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cmn, ax=ax, cmap="Blues", vmin=0, vmax=1,
                cbar=(m == "finetune"), square=True,
                xticklabels=classes, yticklabels=classes)
    ax.set_title(f"resnet18_{m}  (macroF1 {f['macro_f1']:.3f})")
    ax.set_xlabel("predicted"); ax.set_ylabel("true")

plt.tight_layout()
plt.savefig(OUT / "figures" / "13_transfer_confusion.png", dpi=150)
plt.show()

**혼동행렬에서 확인할 것 세 가지**

1. **유리 3×3 블록** — EDA에서 Hue 겹침이 0.5 미만이었으므로 대각선이 진해야 정상입니다.
   `brown-glass ↔ white-glass`(겹침 0.422로 최고)에서 오분류가 남는다면 EDA 예측이 맞은 것입니다.
2. **metal** — `03`의 `none` 실험에서 recall 0.181로 최악이었습니다. 769장으로 소수 클래스가
   아닌데 못 맞혔으니 불균형 탓이 아닙니다. metal 행이 어느 열로 새는지 보세요.
   glass·plastic으로 샌다면 금속 캔의 반사광이 원인일 가능성이 큽니다.
3. **shoes ↔ clothes** — `03`에서 재조정 후 최악이 shoes로 바뀌었습니다.
   다수 클래스를 누르자 shoes가 clothes에 먹힌 것으로 보입니다.

In [ ]:
best_name = compare.sort_values("macro_f1", ascending=False).iloc[0]["model"]
print("최종 선택:", best_name)
print()

if best_name.startswith("resnet18_"):
    r = results[best_name.replace("resnet18_", "")]["final"]
    print(classification_report(r["y_true"], r["y_pred"],
                                target_names=classes, digits=3, zero_division=0))
else:
    print("scratch 모델이 최고 — 05에서 baseline 체크포인트를 사용하세요.")

In [ ]:
# 실험 로그 누적
exp_path = OUT / "metrics" / "experiments.csv"
log = transfer_summary.assign(img_size=IMG_SIZE, epochs=EPOCHS, strategy="sampler")

if exp_path.exists():
    prev = pd.read_csv(exp_path)
    log = pd.concat([prev, log], ignore_index=True)

log.to_csv(exp_path, index=False, encoding="utf-8")
print(log.to_string(index=False))
print("\n체크포인트:", *[p.name for p in sorted((OUT / "models").glob("transfer_*.pt"))])

---

## 다음 단계

`05_eval_report`에서 **여기까지 나온 모델 중 가장 좋은 하나**를 골라
**test split에 단 한 번** 적용합니다. test는 지금까지 한 번도 쓰지 않았습니다.

보고서에 넣을 문장 형태:

> 동일한 데이터·분할·불균형 대응(sampler) 조건에서 scratch CNN은 val `___`%,
> ImageNet 사전학습 ResNet18은 backbone 동결만으로 `___`%를 기록했다.
> backbone을 한 번도 이 데이터로 학습시키지 않았음에도 `___`p 높은 성능을 보인 것은
> ImageNet에서 학습된 저수준 시각 특징이 도메인을 넘어 전이됨을 보여준다.

→ `03P_05_eval_report.ipynb` 로 이동